In [1]:
import pandas as pd
import numpy as np

def standardize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df

def clean_nulls(df):
    df = df.replace(
        ["", "Unknown", "Not_Available", "not_available", "NA", "N/A"],
        np.nan
    )
    return df

def parse_dates(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce", dayfirst=False)
    return df

def parse_datetimes(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


In [2]:
df = pd.read_csv("SLUApplicant_cleaned.csv", dtype=str, encoding="utf-8")

df = standardize_columns(df)
df = clean_nulls(df)

# Fix scientific notation IDs
id_cols = [
    "reference_id", "phone_number", "received_at",
    "created_at", "modified_at"
]

for c in id_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.replace(".0", "", regex=False)

# Date columns
date_cols = ["date_of_birth"]
df = parse_dates(df, date_cols)

datetime_cols = [
    "received_at_datetime",
    "created_at_datetime",
    "modified_at_datetime"
]
df = parse_datetimes(df, datetime_cols)

# Standardize booleans
df["is_global_grad"] = df["is_global_grad"].replace({"Yes": "Yes", "No": "No"})

# Drop duplicates
df = df.drop_duplicates(subset=["reference_id"])

df.to_csv("APPLICANT_clean.csv", index=False)


In [3]:
df = pd.read_csv("SLUConnect_Cleaned.csv", dtype=str, encoding="utf-8")

df = standardize_columns(df)
df = clean_nulls(df)

datetime_cols = ["received_at", "created_at"]
df = parse_datetimes(df, datetime_cols)

df["deposit_status"] = df["deposit_status"].replace({"Yes": "Yes", "No": "No"})
df["i_20_status"] = df["i_20_status"].replace({"Yes": "Yes", "No": "No"})

df = df.drop_duplicates(subset=["reference_id"])

df.to_csv("CONNECT_clean.csv", index=False)


In [4]:
df = pd.read_csv("Copy_of_Outreach_CLEANED.csv", dtype=str, encoding="utf-8")

df = standardize_columns(df)
df = clean_nulls(df)

date_cols = ["appointment_date"]
df = parse_dates(df, date_cols)

bool_cols = [
    "global_grad_indicator",
    "deposit_paid",
    "accepted_admission",
    "i_20_in_slate"
]

for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].replace({"Yes": "Yes", "No": "No"})

df.to_csv("OUTREACH_clean.csv", index=False)


C:\Users\amnaz\AppData\Local\Temp\ipykernel_47712\772064112.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce", dayfirst=False)


In [6]:
df = pd.read_csv("cleaned_SEVIS_data.csv", dtype=str, encoding="utf-8")

df = standardize_columns(df)
df = clean_nulls(df)

date_cols = [
    "status_change_date",
    "date_of_birth",
    "date_of_last_event",
    "program_start_date",
    "program_end_date",
    "initial_session_start_date",
    "current_session_start_date",
    "i-901_transaction_date"
]

df = parse_dates(df, date_cols)

money_cols = [
    "tuition_fees",
    "living_expenses",
    "dependent_expenses",
    "other_costs",
    "students_personal_funds",
    "funds_from_this_school",
    "funds_from_other_sources",
    "i-901_transaction_amount"
]

for c in money_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.drop_duplicates(subset=["sevis_id"])

df.to_csv("SEVIS_clean.csv", index=False)
